# force_q8: flip the default, or not

`GLCUDA_FORCE_Q8` measured **2.08x prefill** on Qwen2.5-0.5B with decode inside
the noise, reproduced across three sessions. That is not enough to make it the
default. Three things decide it, and none of them are measured yet.

| # | question | why it can change the answer |
|---|---|---|
| 1 | VRAM delta per arm | `ffn_down` goes 6.5625 -> 8.5 bpw. Trivial at 0.5B; at 70B it can be the blocker. |
| 2 | decode at 7B | 0.5B has little weight traffic, so decode barely notices. A bigger model may actually pay. |
| 3 | cold vs warm decode | the native k-quant path may be buying cache behaviour that only shows on the first iterations. |

**Decision rule, fixed before the numbers arrive:**

* 7B decode flat or up → flip the default, keep `GLCUDA_NATIVE_KQUANT=1` as the
  escape hatch.
* 7B decode down with no overlap → stays opt-in, and the trade gets documented.

An escape hatch is not a reason to skip this. A wrong default with an escape
hatch is still a wrong default.

### The 7B is not just a bigger 0.5B

The selector is `in_dim % 256`, and the two models land on opposite sides of it:

| | in_dim | % 256 | path |
|---|---:|---:|---|
| 0.5B gate/up/q/k/v/o | 896 | 128 | Q8_0 → **GEMM** |
| 0.5B down | 4864 | 0 | native → GEMV |
| **7B everything** | 3584 / 18944 | **0** | native → **GEMV** |

At 0.5B only `down` misses the GEMM. At 7B **nothing gets it** — 3584 is
14×256. So 0.5B was the mild case by accident, the 7B baseline should be far
worse at prefill, and the VRAM delta covers the whole weight set rather than
one projection. Which is exactly why VRAM is on this list.

### Disk is the binding constraint

Each arm stages its own `.glcache` — the path carries the weight-format policy,
or the forced arm would be handed the native weights it exists to avoid. For
the 7B that is roughly

    model 4.7 + native cache 4.7 + q8 cache 8.7  =  18 GB

against ~20 GB of `/kaggle/working`. Each model's artifacts are deleted once it
is done, and a free-space gate runs before each one rather than letting a run
die halfway through.

Same guards as the probe: interleaved arms, one warmup sweep discarded, the
engine banner checked, the overlap rule instead of a percentage bar, and this
notebook's own stamp compared against the branch's copy.

## 1 · setup

In [ ]:
REPO_URL, BRANCH, GH_TOKEN = "https://github.com/gwenland-org/gwenland-ai.git", "glbench-vs-llamacpp", ""

# (label, HF repo id, quant, iters). Filenames are RESOLVED from the HF API,
# not written here: Qwen's own 7B repo ships Q4_K_M as two shards
# (-00001-of-00002), a hardcoded single name 404s, and this notebook's silent
# skip cost a whole run before anyone noticed. glcore's GGUF loader has no
# split-file support, so a sharded repo is unusable and the resolver says so
# instead of guessing.
#
# 3B is a middle rung: 2048 and 11008 are both multiples of 256, so it has the
# SAME structural property as the 7B (every matmul on the native path) at a
# third the disk and time. It is insurance -- if the 7B does not fit, the
# question still gets an answer above 0.5B.
MODELS = [
    ("0.5B", "Qwen/Qwen2.5-0.5B-Instruct-GGUF", "q4_k_m", 10),
    ("3B", "bartowski/Qwen2.5-3B-Instruct-GGUF", "q4_k_m", 8),
    ("7B", "bartowski/Qwen2.5-7B-Instruct-GGUF", "q4_k_m", 5),
]
GEN_TOKENS, WARMUP = 128, 3
AB_ENV, AB_BANNER = {"GLCUDA_FORCE_Q8": "1"}, "GLCUDA_FORCE_Q8:"
AB_MARKER = ("glcuda/src/loader.rs", "GLCUDA_FORCE_Q8")
AB_REPEATS, AB_WARMUP = 3, 1
NB_STAMP = "c58c4ee0"

import os, re, json, time, shutil, statistics, subprocess
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
REPO_DIR, OUT_DIR = os.path.join(WORK, "gwenland-ai"), WORK
os.makedirs(WORK, exist_ok=True)

def sh(cmd, cwd=None, timeout=7200, env=None):
    e = dict(os.environ); e.update(env or {})
    try:
        p = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True,
                           timeout=timeout, env=e, stdin=subprocess.DEVNULL)
        return p.returncode, p.stdout, p.stderr
    except subprocess.TimeoutExpired: return 124, "", "timeout"
    except Exception as ex: return 125, "", f"{type(ex).__name__}: {ex}"

rc, out, _ = sh(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], timeout=60)
GPUS = [l.strip() for l in out.splitlines() if l.strip()] if rc == 0 else []
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print(f"gpu     {len(GPUS)}x {GPUS[0] if GPUS else 'NONE'} -> pinned dev 0")

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    rc1, _, e1 = sh(["git", "fetch", "--depth", "50", "origin", BRANCH], cwd=REPO_DIR)
    rc2, _, e2 = sh(["git", "reset", "--hard", "FETCH_HEAD"], cwd=REPO_DIR)
    if rc1 or rc2: print("        ⛔ refresh FAILED:", (e1 or e2)[:150])
else:
    url = REPO_URL.replace("https://", f"https://{GH_TOKEN}@") if GH_TOKEN else REPO_URL
    sh(["git", "clone", "--depth", "50", "--branch", BRANCH, url, REPO_DIR])
_, o, _ = sh(["git", "log", "--oneline", "-1"], cwd=REPO_DIR)
COMMIT = o.strip()
print(f"commit  {COMMIT[:60]}")

_nb = os.path.join(REPO_DIR, "notebooks", "force_q8_decision.ipynb")
if os.path.exists(_nb):
    _m = re.search(r'NB_STAMP = .([0-9a-f]{8}).', open(_nb, encoding="utf-8", errors="replace").read())
    if _m and _m.group(1) != NB_STAMP:
        print(f"        ⛔ THIS NOTEBOOK IS STALE: running {NB_STAMP}, branch has {_m.group(1)}.")
        print(f"           Re-import it. Verdicts below are the old notebook's.")
    elif _m:
        print(f"notebook {NB_STAMP} (matches branch)")

_src = os.path.join(REPO_DIR, *AB_MARKER[0].split("/"))
HAVE = os.path.exists(_src) and AB_MARKER[1] in open(_src, errors="replace").read()
if not HAVE:
    print(f"        ⛔ {AB_MARKER[1]} absent -- both arms would be identical")

if shutil.which("cargo") is None:
    os.system("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal >/dev/null 2>&1")
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]

def free_gb(p=WORK):
    s = os.statvfs(p); return s.f_bavail * s.f_frsize / 1e9
print(f"disk    {free_gb():.1f} GB free")


## 2 · build + models

⛔ Disk is the real constraint here. Each arm stages its own `.glcache` next to
the GGUF (the cache path carries the weight-format policy, or the forced arm
would be handed the native weights it exists to avoid). For a 7B that is the
model plus two staged copies.

In [ ]:
GL_BIN = os.path.join(REPO_DIR, "target/release/glbench")
CACHE = os.path.join(WORK, "glbench-bin-cache"); man_p = os.path.join(CACHE, "manifest.json")
restored = False
if os.path.exists(man_p):
    try:
        if json.load(open(man_p)).get("commit") == COMMIT and os.path.exists(os.path.join(CACHE, "glbench")):
            os.makedirs(os.path.dirname(GL_BIN), exist_ok=True)
            shutil.copy2(os.path.join(CACHE, "glbench"), GL_BIN); os.chmod(GL_BIN, 0o755); restored = True
    except Exception: pass
if not restored:
    t = time.time()
    rc, o, e = sh(["cargo", "build", "--release", "-p", "glbench"], cwd=REPO_DIR)
    assert rc == 0, (o + e)[-3000:]
    os.makedirs(CACHE, exist_ok=True)
    shutil.copy2(GL_BIN, os.path.join(CACHE, "glbench"))
    json.dump({"commit": COMMIT}, open(man_p, "w"))
    print(f"build   {(time.time()-t)/60:.1f} min")
else:
    print("build   cached")

import urllib.request

def resolve(repo, quant):
    # The single .gguf for `quant` in `repo`, or a reason it cannot be used.
    try:
        with urllib.request.urlopen(
                f"https://huggingface.co/api/models/{repo}", timeout=60) as r:
            sibs = [s["rfilename"] for s in json.load(r).get("siblings", [])]
    except Exception as ex:
        return None, f"HF API unreachable: {type(ex).__name__}: {ex}"
    hits = [f for f in sibs if f.lower().endswith(".gguf") and quant in f.lower()]
    if not hits:
        return None, f"no .gguf matching {quant!r} among {len(sibs)} files"
    shards = [f for f in hits if re.search(r"-\d{5}-of-\d{5}\.gguf$", f)]
    if shards and len(shards) == len(hits):
        return None, (f"only sharded: {shards[0]} (+{len(shards)-1} more). "
                      f"glcore's GGUF loader cannot read split files.")
    whole = [f for f in hits if f not in shards]
    return sorted(whole, key=len)[0], None

PATHS = {}
for label, repo, quant, _ in MODELS:
    fname, why = resolve(repo, quant)
    if not fname:
        print(f"{label:5s}   ⛔ SKIPPED -- {why}")
        continue
    p = os.path.join(WORK, fname)
    if not (os.path.exists(p) and os.path.getsize(p) > 1e7):
        print(f"downloading {label}: {fname}")
        rc, _, e = sh(["curl", "-fL", "--retry", "3", "-o", p,
                       f"https://huggingface.co/{repo}/resolve/main/{fname}"], timeout=3600)
        if rc != 0:
            print(f"{label:5s}   ⛔ curl exit {rc}: {(e or '').strip()[:160]}")
    if os.path.exists(p) and os.path.getsize(p) > 1e7:
        PATHS[label] = p
        print(f"{label:5s}   {os.path.getsize(p)/1e9:.2f} GB  {fname}")
    else:
        got = os.path.getsize(p) if os.path.exists(p) else 0
        print(f"{label:5s}   ⛔ MISSING -- got {got} bytes, skipping")
print(f"disk    {free_gb():.1f} GB free after downloads")


## 2.5 · correctness gate

⛔ **A hard STOP, not a report.** Everything below this cell is a wall-clock
measurement, and a wall-clock number from code that computes the wrong answer
is worse than no number — it looks like a result.

Three things are checked, in order:

1. **The checkout actually contains the changes.** The engine changes live in
   the working tree until they are pushed; a notebook fetching a stale branch
   would validate *and* benchmark the old engine, then report the number
   against the new one. Each marker is a string that exists only in the new code.
2. **The GPU suites pass** — `parity`, `graph_replay`, `forward`.
3. **They did not silently skip.** This is the trap this repo has already been
   bitten by: every device test prints `SKIP: no CUDA driver/device` and still
   reports `ok`, so a green suite on a machine without a GPU proves nothing. A
   skip here means the T4 was never visible, and the gate fails.

`graph_replay` is the new one: it decodes the same tokens twice — once through
individual kernel launches, once through the captured CUDA graph — and asserts
the logits match **bit for bit**. If that fails, do not loosen it to a
tolerance; the two paths run identical kernels over identical buffers, so a
difference is a real defect in the capture.

In [ ]:
# --- 2.5 correctness gate -------------------------------------------------
# Raises on failure so "Run All" halts here. CORRECTNESS_OK is also consulted
# by the verdict cell, for anyone running cells one at a time.
CORRECTNESS_OK, GATE_NOTES = False, []

# (1) Is this checkout the code we mean to test? The engine changes live in
# the working tree until they are pushed, and a stale branch would benchmark
# the OLD engine while attributing the number to the new one.
GATE_MARKERS = [
    ("glcuda/src/runner.rs",         "gemv_w_pre",       "Fix 1 - shared q/k/v quantize"),
    ("glcuda/src/runner.rs",         "qn.dptr, q_ptr",   "Fix 2 - batched per-head norm"),
    # NB: "rms_norm_rows(" would NOT discriminate -- prefill has always called it.
    # The marker has to be the decode-side call, which is new.
    ("glcuda/src/driver.rs",         "graphs_available", "graph fallback (optional symbols)"),
    ("glcuda/tests/graph_replay.rs", "replays_see_updated_position", "graph replay suite"),
]
stale = []
for path, needle, what in GATE_MARKERS:
    fp = os.path.join(REPO_DIR, path)
    txt = open(fp, encoding="utf-8", errors="replace").read() if os.path.exists(fp) else ""
    print("  [%-7s] %s" % ("ok" if needle in txt else "MISSING", what))
    if needle not in txt:
        stale.append("%s: %r not found in %s" % (what, needle, path))
if stale:
    raise SystemExit(
        "\n⛔ STALE CHECKOUT at %s -- the engine changes are not on this branch.\n   %s\n\n"
        "   Commit and push them, then re-run from cell 1. Benchmarking this checkout\n"
        "   would measure the OLD engine and report it as the new one."
        % (COMMIT, "\n   ".join(stale))
    )

# (2) + (3) Run each GPU suite; insist it neither failed nor silently skipped.
# The skip check is the one that matters: every device test prints
# "SKIP: no CUDA driver/device" and still reports ok, so a green suite on a
# machine without a visible GPU proves exactly nothing.
SUITES = ["parity", "graph_replay", "forward"]
gate_fail = []
for suite in SUITES:
    t = time.time()
    rc, o, e = sh(["cargo", "test", "-p", "glcuda", "--release", "--test", suite,
                   "--", "--test-threads=1", "--nocapture"], cwd=REPO_DIR, timeout=3600)
    hay = o + "\n" + e
    m = re.search(r"test result: (\w+)\. (\d+) passed; (\d+) failed", hay)
    skips = len(re.findall(r"SKIP: no CUDA driver/device", hay))
    passed = int(m.group(2)) if m else 0
    failed = int(m.group(3)) if m else -1
    if rc != 0 or not m or failed != 0:
        gate_fail.append("%s: rc=%s, %s failed\n%s" % (suite, rc, failed, hay[-2000:]))
        verdict = "FAILED"
    elif skips:
        gate_fail.append(
            "%s: %d test(s) skipped for lack of a CUDA device -- the suite is "
            "green but tested nothing" % (suite, skips))
        verdict = "SKIPPED x%d" % skips
    else:
        verdict = "pass"
    print("  %-13s %-12s %3d passed  %5.0fs" % (suite, verdict, passed, time.time() - t))
    GATE_NOTES.append((suite, verdict, passed, skips))

if gate_fail:
    raise SystemExit("\n⛔ CORRECTNESS GATE FAILED -- do not read any number below.\n\n"
                     + "\n\n".join(gate_fail))

CORRECTNESS_OK = True
print("\n✅ gate passed on real hardware: parity + graph replay + end-to-end forward.")
print("   Fix 1, Fix 2 and the graph fallback are verified now, not assumed.")

## 3 · run

Per model, per arm: interleaved, one warmup sweep discarded. Recorded for every
run — prefill, **warm decode**, **cold decode**, and **VRAM reserved**.

Cold decode comes from glbench's own `measurements.cold` block, which is
separate from the measured iterations; warm decode from `measurements.iterations`.

In [ ]:
def tps(iters, phase):
    v = []
    for it in iters or []:
        ms = it.get(f"{phase}_ms", 0.0)
        n = it.get("prompt_tokens" if phase == "prefill" else "generated_tokens", 0)
        if ms > 0 and n: v.append(n / (ms / 1e3))
    return statistics.median(v) if v else None

def one(model_path, iters, env, tag):
    out = os.path.join(OUT_DIR, f"d_{tag}.json")
    cmd = [GL_BIN, "run", "--engine", "glcuda", "--model", model_path,
           "--tokens", str(GEN_TOKENS), "--warmup", str(WARMUP),
           "--iters", str(iters), "--out", out]
    rc, o, e = sh(cmd, cwd=REPO_DIR, timeout=7200, env=env)
    hay = o + "\n" + e
    vram = re.search(r"(\d+) MiB VRAM reserved", hay)
    rec = {"rc": rc, "vram": int(vram.group(1)) if vram else None,
           "banner": AB_BANNER in hay, "prefill": None, "warm": None, "cold": None}
    try:
        m = json.load(open(out))["measurements"]
        rec["prefill"] = tps(m.get("iterations"), "prefill")
        rec["warm"] = tps(m.get("iterations"), "decode")
        rec["cold"] = tps(m.get("cold"), "decode")
    except Exception as ex:
        rec["err"] = f"{type(ex).__name__}: {ex}"
    return rec

def cleanup(label):
    p = PATHS.get(label)
    if not p:
        return
    freed = 0.0
    for f in [p, p + ".glcache", p + ".q8.glcache"]:
        if os.path.exists(f):
            freed += os.path.getsize(f) / 1e9
            try: os.remove(f)
            except OSError: pass
    print(f"  cleaned {label}: freed {freed:.1f} GB, {free_gb():.1f} GB now free")

R = {}
for mi, (label, _, _, iters) in enumerate(MODELS):
    if label not in PATHS or not HAVE:
        print(f"\n== {label}: SKIPPED ==")
        continue
    # Model + native cache + q8 cache, the last roughly 1.85x the model because
    # Q4_K at 4.5 bpw becomes Q8_0 at 8.5. Both caches must exist at once, or
    # the arms cannot interleave.
    need = os.path.getsize(PATHS[label]) / 1e9 * 3.85
    if free_gb() < need:
        print(f"\n== {label}: SKIPPED -- needs ~{need:.1f} GB, {free_gb():.1f} GB free ==")
        print("   Free space, or drop a model from MODELS.")
        continue
    print(f"\n== {label} ==  (needs ~{need:.1f} GB, {free_gb():.1f} GB free)")
    R[label] = {"base": [], "var": []}
    for r in range(AB_REPEATS + AB_WARMUP):
        for arm, env in (("base", {}), ("var", AB_ENV)):
            rec = one(PATHS[label], iters, env, f"{label}_{arm}_{r}")
            kept = r >= AB_WARMUP
            if kept: R[label][arm].append(rec)
            print(f"  {label:5s} {arm:4s} r{r} rc={rec['rc']}  "
                  f"prefill={rec['prefill'] or 0:7.1f}  warm={rec['warm'] or 0:6.1f}  "
                  f"cold={rec['cold'] or 0:6.1f}  vram={rec['vram']}MiB"
                  + ("" if kept else "   (warmup, discarded)"))
    b = any(x["banner"] for x in R[label]["var"])
    print(f"  banner seen in var arm: {b}" + ("" if b else "   ⛔ arms may be identical"))
    if mi + 1 < len(MODELS):
        cleanup(label)   # free this model's footprint before the next needs it


## 4 · verdict

In [ ]:
def spread(recs, key):
    v = sorted(x[key] for x in recs if x.get(key) is not None)
    return (v[0], statistics.median(v), v[-1]) if v else (None, None, None)

def overlap(a, b):
    return not (a[0] > b[2] or a[2] < b[0])

SUMMARY = []
for label in R:
    row = {"model": label}
    for key in ("prefill", "warm", "cold", "vram"):
        bb, vv = spread(R[label]["base"], key), spread(R[label]["var"], key)
        row[key] = (bb, vv)
    SUMMARY.append(row)

for row in SUMMARY:
    print(f"\n== {row['model']} ==")
    for key, unit in (("prefill", "tok/s"), ("warm", "tok/s"), ("cold", "tok/s"), ("vram", "MiB")):
        bb, vv = row[key]
        if bb[1] is None or vv[1] is None:
            print(f"  {key:8s} UNMEASURED"); continue
        d = 100 * (vv[1] - bb[1]) / bb[1]
        ov = overlap(bb, vv)
        print(f"  {key:8s} base {bb[0]:8.1f}-{bb[2]:<8.1f} med {bb[1]:8.1f} | "
              f"q8 {vv[0]:8.1f}-{vv[2]:<8.1f} med {vv[1]:8.1f} | {d:+6.1f}% "
              f"{unit}  {'(overlap)' if ov else '(no overlap)'}")

# The rule, applied to the 7B row.
# The banner is a gate, not a remark. An arm that never set the flag is the
# same arm as the baseline, and its numbers are a comparison of nothing --
# which is exactly what an ignored env= produced once.
BANNER_OK = {lab: any(x["banner"] for x in R[lab]["var"]) for lab in R}
VRAM_MOVED = {}
for lab in R:
    b, v = spread(R[lab]["base"], "vram"), spread(R[lab]["var"], "vram")
    VRAM_MOVED[lab] = (b[1] is not None and v[1] is not None and v[1] != b[1])
for lab in R:
    if not BANNER_OK[lab]:
        print(f"  ⛔ {lab}: force_q8 banner never appeared -- the arms are identical")
    elif not VRAM_MOVED[lab]:
        print(f"  ⛔ {lab}: VRAM identical across arms. Requantizing k-quants to "
              f"Q8_0 must raise it; unchanged means it did not happen.")

if not globals().get("CORRECTNESS_OK"):
    raise SystemExit(
        "⛔ the correctness gate (cell 2.5) did not pass in this session. "
        "These timings describe an engine nothing has verified."
    )

DECISION = "UNDECIDED -- no 7B data"
big = next((r for r in SUMMARY if r["model"] == "7B"), None)
if big and not (BANNER_OK.get("7B") and VRAM_MOVED.get("7B")):
    DECISION = ("INVALID -- the 7B arms did not differ (no banner, or VRAM "
                "unchanged). Nothing below is a measurement of force_q8.")
elif big:
    bb, vv = big["warm"]
    if bb[1] is None or vv[1] is None:
        DECISION = "UNDECIDED -- 7B decode unmeasured"
    else:
        d = 100 * (vv[1] - bb[1]) / bb[1]
        if overlap(bb, vv) or d > 0:
            DECISION = (f"FLIP THE DEFAULT -- 7B warm decode {d:+.1f}% "
                        f"{'(arms overlap)' if overlap(bb, vv) else '(improved)'}; "
                        f"keep GLCUDA_NATIVE_KQUANT=1 as the escape hatch")
        else:
            DECISION = (f"STAY OPT-IN -- 7B warm decode {d:+.1f}% with no overlap. "
                        f"A real trade: document it, do not default it")
    cb, cv = big["cold"]
    if cb[1] and cv[1] and not overlap(cb, cv):
        DECISION += f" · NOTE cold decode {100*(cv[1]-cb[1])/cb[1]:+.1f}% (no overlap)"
    vb, vv2 = big["vram"]
    if vb[1] and vv2[1]:
        DECISION += f" · VRAM {vv2[1]-vb[1]:+.0f} MiB ({100*(vv2[1]-vb[1])/vb[1]:+.1f}%)"

print("\n" + "=" * 66)
print(DECISION)
print("=" * 66)


## 5 · report

In [ ]:
L = [f"# force_q8 decision · {time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime())}",
     f"`{COMMIT}`  ·  {len(GPUS)}x {GPUS[0] if GPUS else '-'}  ·  notebook {NB_STAMP}", "",
     f"**{DECISION}**", "",
     f"{AB_REPEATS} repeats per arm, interleaved, {AB_WARMUP} warmup sweep discarded. "
     f"Ranges are min-max; the verdict turns on whether they overlap, not on the "
     f"percentage.", ""]
for row in SUMMARY:
    L += [f"## {row['model']}", "", "| metric | baseline | force_q8 | change | |",
          "|---|---|---|---:|---|"]
    for key, unit in (("prefill", "tok/s"), ("warm", "decode tok/s"),
                      ("cold", "cold decode tok/s"), ("vram", "MiB")):
        bb, vv = row[key]
        if bb[1] is None or vv[1] is None:
            L.append(f"| {key} | - | - | - | unmeasured |"); continue
        d = 100 * (vv[1] - bb[1]) / bb[1]
        L.append(f"| {unit} | {bb[0]:.1f}-{bb[2]:.1f} (med {bb[1]:.1f}) | "
                 f"{vv[0]:.1f}-{vv[2]:.1f} (med {vv[1]:.1f}) | {d:+.1f}% | "
                 f"{'overlap' if overlap(bb, vv) else 'no overlap'} |")
    L.append("")
L += ["## rule", "",
      "* 7B decode flat or up -> flip the default, keep `GLCUDA_NATIVE_KQUANT=1`.",
      "* 7B decode down with no overlap -> stays opt-in, trade documented.", "",
      "Fixed before the numbers arrived. An escape hatch is not a reason to skip "
      "the measurement: a wrong default with an escape hatch is still a wrong "
      "default.", ""]
p = os.path.join(OUT_DIR, "FORCE_Q8_DECISION.md")
open(p, "w", encoding="utf-8").write("\n".join(L))
print(p)
